## Imports

In [1]:
from alphagenome.data import genome
from alphagenome.visualization import plot_components
from alphagenome_research.model import dna_model
from alphagenome.models import dna_client
import matplotlib.pyplot as plt
import pandas as pd

## Local Model

In [3]:
# Load model using persistent HF auth and local FASTA override
import os
from pathlib import Path
from getpass import getpass
from huggingface_hub import whoami, login
from alphagenome_research.model import dna_model
from alphagenome.models import dna_model as public_dna_model


def ensure_hf_auth(allow_prompt=True):
    try:
        info = whoami()
        print(f"HF already authenticated as: {info.get('name', 'unknown')}")
        return
    except Exception:
        pass

    token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
    if token:
        login(token=token, add_to_git_credential=False, skip_if_logged_in=True)
        info = whoami()
        print(f"HF authenticated via environment token as: {info.get('name', 'unknown')}")
        return

    if allow_prompt:
        try:
            token = getpass("Enter Hugging Face token (input hidden): ").strip()
        except Exception:
            token = ""

        if token:
            login(token=token, add_to_git_credential=False, skip_if_logged_in=True)
            info = whoami()
            print(f"HF authenticated via interactive prompt as: {info.get('name', 'unknown')}")
            return

    raise RuntimeError(
        "No Hugging Face credentials found. Use one of:\n"
        "1) Run `hf auth login` in a terminal\n"
        "2) Set HF_TOKEN (or HUGGINGFACEHUB_API_TOKEN)\n"
        "3) Re-run this cell and paste a token when prompted"
    )


def resolve_local_human_fasta_path():
    candidates = [
        os.getenv("ALPHAGENOME_FASTA"),
        "/data/Dcode/gaetano/repos/fasta_files/hg38.fa",
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return candidate
    return None


def resolve_local_mouse_fasta_path():
    candidates = [
        os.getenv("ALPHAGENOME_MOUSE_FASTA"),
        "/data/Dcode/gaetano/repos/fasta_files/mm10.fa",
        "/data/Dcode/gaetano/repos/fasta_files/mm39.fa",
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return candidate
    return None


ensure_hf_auth(allow_prompt=True)

human_fasta = resolve_local_human_fasta_path()
if not human_fasta:
    raise FileNotFoundError(
        "No local human FASTA found. Set ALPHAGENOME_FASTA to a local GRCh38 FASTA path, e.g.\n"
        "export ALPHAGENOME_FASTA=/path/to/GRCh38.p13.genome.fa"
    )

mouse_fasta = resolve_local_mouse_fasta_path()
if mouse_fasta:
    print(f"Using local mouse FASTA: {mouse_fasta}")
else:
    print(
        "No local mouse FASTA found; configuring mouse organism without local FASTA. "
        "Predictions on human intervals still work."
    )

print(f"Using local human FASTA: {human_fasta}")
organism_settings = {
    public_dna_model.Organism.HOMO_SAPIENS: dna_model.OrganismSettings(
        fasta_path=human_fasta,
    ),
    public_dna_model.Organism.MUS_MUSCULUS: dna_model.OrganismSettings(
        fasta_path=mouse_fasta,
    ) if mouse_fasta else dna_model.OrganismSettings(),
}

model = dna_model.create_from_huggingface("fold-1", organism_settings=organism_settings)
print("Model loaded successfully.")

HF already authenticated as: tanoManzo
No local mouse FASTA found; configuring mouse organism without local FASTA. Predictions on human intervals still work.
Using local human FASTA: /data/Dcode/gaetano/repos/fasta_files/hg38.fa


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Model loaded successfully.


## Load Data

### HepG2 Enhancer Training Data

In [ ]:
# HepG2 Enhancer
path_training_data = "/vf/users/Dcode/gaetano/FootPrinting/data/input_training_trednet/enhancers_dhs_control"
path_positive_data = f"{path_training_data}/HepG2_Enhancer_DHS_positive_1kb.bed"
path_control_data = f"{path_training_data}/HepG2_Enhancer_DHS_control_1kb.bed"

bed_cols = ["chrom", "start", "end"]
positive_hepg2_enh = pd.read_csv(path_positive_data, sep="\t", header=None, names=bed_cols)
control_hepg2_enh = pd.read_csv(path_control_data, sep="\t", header=None, names=bed_cols)


positive_hepg2_enh.head(), control_hepg2_enh.head()

(  chrom    start      end
 0  chr1   190392   191392
 1  chr1   726724   727724
 2  chr1  1006170  1007170
 3  chr1  1017069  1018069
 4  chr1  1079090  1080090,
   chrom   start     end
 0  chr1   19290   20290
 1  chr1  108689  109689
 2  chr1  189280  190280
 3  chr1  592693  593693
 4  chr1  603750  604750)

### Convert into Alpha Genome Intervals

In [5]:
sample_size = 5

positive_intervals = []
for index, row in positive_hepg2_enh.sample(sample_size).iterrows():
    positive_intervals.append(genome.Interval(chromosome=row['chrom'], start=row['start'], end=row['end']))

control_intervals = []
for index, row in control_hepg2_enh.sample(sample_size).iterrows():
    control_intervals.append(genome.Interval(chromosome=row['chrom'], start=row['start'], end=row['end']))

## Predictions 

In [6]:
from tqdm.auto import tqdm

def predict_intervals(intervals, sequence_length=dna_client.SEQUENCE_LENGTH_16KB):
    all_outputs = []
    for iv in tqdm(intervals, desc="Predicting intervals"):
        iv = iv.resize(sequence_length)  # or 1MB if needed
        out = model.predict_interval(
            interval=iv,
            requested_outputs=list(dna_model.OutputType),
            ontology_terms=None,
        )
        all_outputs.append(out)
    return all_outputs

all_outputs_positive = predict_intervals(positive_intervals)
all_outputs_control = predict_intervals(control_intervals)

Predicting intervals:   0%|          | 0/5 [00:00<?, ?it/s]

Predicting intervals:   0%|          | 0/5 [00:00<?, ?it/s]

In [7]:
# 2) Iterate through every output type and print what's present
outputs = all_outputs_positive[0]
for ot in dna_model.OutputType:
    item = outputs.get(ot)
    if item is not None:
        vals = getattr(item, "values", None)
        shape = getattr(vals, "shape", None)
        print(f"{ot.name}: {type(item).__name__}, shape={shape}")

ATAC: TrackData, shape=(16384, 167)
CAGE: TrackData, shape=(16384, 546)
DNASE: TrackData, shape=(16384, 305)
RNA_SEQ: TrackData, shape=(16384, 667)
CHIP_HISTONE: TrackData, shape=(128, 1116)
CHIP_TF: TrackData, shape=(128, 1617)
SPLICE_SITES: TrackData, shape=(16384, 4)
SPLICE_SITE_USAGE: TrackData, shape=(16384, 734)
SPLICE_JUNCTIONS: JunctionData, shape=(152, 367)
CONTACT_MAPS: TrackData, shape=(8, 8, 28)
PROCAP: TrackData, shape=(16384, 12)


In [21]:
all_outputs_positive[0].rna_seq[0].values.shape

(1, 667)